# BAREC Dataset Arabic Processing Configuration Demo

This notebook demonstrates how different Arabic text processing configurations affect the BAREC dataset examples. We'll test various normalization, diacritization, and morphological processing options and visualize the results.

## Setup and Configuration

In [ ]:
import sys
import os
import logging
import matplotlib.pyplot as plt
import pandas as pd
from typing import Dict, List, Any
import torch
from PIL import Image
import numpy as np
from collections import defaultdict

# Add the project root to Python path
sys.path.append('/home/bens/pixel')

from src.pixel.data.datasets.barec_dataset import BARECDataset
from src.pixel.data.rendering import PangoCairoTextRenderer
from src.pixel import Modality, get_transforms
from src.pixel.data.processing.arabic_sentence_processor import (
    ProcessingConfig, 
    ArabicSentenceProcessor,
    OrthographicFormat,
    DiacriticFormat,
    MorphologicalScheme,
    EncodingScheme,
    create_default_config,
    create_normalized_config,
    create_diacritized_config,
    create_morphological_config,
    create_buckwalter_config
)

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print("✅ All imports successful!")

## Define Processing Configurations

Let's create various processing configurations to test:

In [ ]:
from src.pixel.data.processing.experiment_configs import ALL_CONFIGS
# Define various processing configurations for testing
processing_configs = ALL_CONFIGS

print(f"Created {len(processing_configs)} processing configurations:")
for name in processing_configs.keys():
    print(f"  - {name}")

## Register Configurations and Test Processing

First, let's test the Arabic sentence processor with some sample sentences:

In [ ]:

# Test sample sentences
sample_sentences = [
    "هَـــلْ ذَهَبْتَ إِلَى المَكْتَبَةِ؟",  # With diacritics and tatweel
    "الولايات المتحدة الأمريكية دولة كبيرة",  # Complex phrase
    "كَتَبَ الطالِبُ الدَّرْسَ بِعِنايَةٍ",  # Diacritized sentence
    "في هذا اليوم الجميل نذهب إلى المدرسة"  # Simple sentence
]

print("\n🔍 Testing sentence processing with different configurations:")
print("=" * 80)

# Test each configuration with the first sample sentence
test_sentence = sample_sentences[0]
print(f"Original: {test_sentence}")
print("-" * 50)

processing_results = {}
for config_name, config in processing_configs.items():
    try:
        processor = ArabicSentenceProcessor(config)
        result = processor.process(test_sentence)
        processing_results[config_name] = result
        print(f"{config_name:15}: {result}")
    except Exception as e:
        print(f"{config_name:15}: ERROR - {e}")
        processing_results[config_name] = f"ERROR: {e}"

## Load BAREC Dataset with Different Configurations

Now let's load the BAREC dataset with different processing configurations:

In [ ]:
# Configuration for dataset loading
DATASET_CONFIG = {
    "dataset_name": "CAMeL-Lab/BAREC-Shared-Task-2025-sent",
    "split": "validation",  # Use train split for examples
    "max_seq_length": 256,
    "renderer_path": "Team-PIXEL/pixel-base",
    "num_samples": 10  # Load only first 10 samples for demo
}

print(f"📊 Loading BAREC dataset: {DATASET_CONFIG['dataset_name']}")
print(f"Split: {DATASET_CONFIG['split']}, Max length: {DATASET_CONFIG['max_seq_length']}")

In [ ]:
# Load renderer for image generation
print("🔧 Loading PIXEL renderer...")
renderer = PangoCairoTextRenderer.from_pretrained(DATASET_CONFIG["renderer_path"], font_features="calt=0,init=0,medi=0,fina=0,liga=0,clig=0")
renderer.max_seq_length = DATASET_CONFIG["max_seq_length"]

# Set up transforms
transforms = get_transforms(
    do_resize=True,
    size=(renderer.pixels_per_patch, renderer.pixels_per_patch * renderer.max_seq_length),
)

print(f"   ✅ Renderer loaded: {renderer.pixels_per_patch}px per patch")
print(f"   ✅ Transform size: {transforms.transforms[0].size if hasattr(transforms.transforms[0], 'size') else 'N/A'}")

In [ ]:
# Load datasets with different processing configurations
datasets = {}
sample_data = {}

# Select a subset of configurations for dataset loading (to avoid loading too many)
selected_configs = ["arabic-default", "arabic-norm-dediac",]

print("📦 Loading datasets with different processing configurations...")

for config_name in selected_configs:
    try:
        print(f"\n   Loading: {config_name}")
        
        # Create dataset
        dataset = BARECDataset(
            dataset_name=DATASET_CONFIG["dataset_name"],
            processor=renderer,
            modality=Modality.IMAGE,
            max_seq_length=DATASET_CONFIG["max_seq_length"],
            split=DATASET_CONFIG["split"],
            transforms=transforms,
            processing_config_name=config_name if config_name != "original" else None,
        )
        
        datasets[config_name] = dataset
        
        # Extract sample data for analysis
        sample_data[config_name] = {
            'sentences': [dataset.examples[i].sentence for i in range(min(DATASET_CONFIG["num_samples"], len(dataset)))],
            'labels': [dataset.examples[i].label for i in range(min(DATASET_CONFIG["num_samples"], len(dataset)))],
            'ids': [dataset.examples[i].id for i in range(min(DATASET_CONFIG["num_samples"], len(dataset)))]
        }
        
        print(f"   ✅ Loaded {len(dataset)} examples")
        
    except Exception as e:
        print(f"   ❌ Error loading {config_name}: {e}")
        datasets[config_name] = None
        sample_data[config_name] = None

print(f"\n✅ Successfully loaded {len([d for d in datasets.values() if d is not None])} datasets")

## Compare Processing Results

Let's create a comparison table showing how the same sentences are processed differently:

In [ ]:
# Create comparison DataFrame
comparison_data = []

for i in range(min(5, DATASET_CONFIG["num_samples"])):
    row = {'Example': i + 1}
    
    for config_name in selected_configs:
        if sample_data[config_name] is not None:
            sentence = sample_data[config_name]['sentences'][i]
            label = sample_data[config_name]['labels'][i]
            row[f'{config_name}'] = sentence
            row[f'{config_name}_label'] = label
    
    comparison_data.append(row)

# Create comparison DataFrame
comparison_df = pd.DataFrame(comparison_data)

print("📋 Sentence Processing Comparison (First 5 Examples)")
print("=" * 100)

for i, row in comparison_df.iterrows():
    print(f"\n🔍 Example {row['Example']}:")
    print("-" * 50)
    
    for config_name in selected_configs:
        if config_name in row and pd.notna(row[config_name]):
            sentence = row[config_name]
            label = row.get(f'{config_name}_label', 'N/A')
            print(f"{config_name:15}: {sentence} (Label: {label})")

## Visualize Rendered Images

Let's visualize how the different processing configurations affect the rendered images:

In [ ]:
def plot_rendered_images(datasets, example_idx=0, max_configs=4):
    """
    Plot rendered images for different processing configurations
    """
    configs_to_plot = list(datasets.keys())[:max_configs]
    
    fig, axes = plt.subplots(len(configs_to_plot), 1)
    if len(configs_to_plot) == 1:
        axes = [axes]
    
    for i, config_name in enumerate(configs_to_plot):
        if datasets[config_name] is not None:
            try:
                # Get the processed example
                example = datasets[config_name][example_idx]
                pixel_values = example['pixel_values']
                
                # Convert to numpy for plotting
                if isinstance(pixel_values, torch.Tensor):
                    if pixel_values.dim() == 3:  # [C, H, W]
                        if pixel_values.shape[0] == 3:  # RGB
                            img_array = pixel_values.permute(1, 2, 0).numpy()
                        else:  # Grayscale
                            img_array = pixel_values[0].numpy()
                    else:
                        img_array = pixel_values.numpy()
                else:
                    img_array = pixel_values
                
                # Plot the image
                # Optionally crop the image array to a given width
                crop_width = 200
                if crop_width is not None and img_array.shape[1] > crop_width:
                    img_array = img_array[:, :crop_width] if len(img_array.shape) == 2 else img_array[:, :crop_width, :]

                axes[i].imshow(img_array, cmap='gray' if len(img_array.shape) == 2 else None)
                
                # Add title with configuration name and processed sentence
                sentence = datasets[config_name].examples[example_idx].sentence
                title = f"{config_name}\n"
                axes[i].set_title(title, fontsize=10, wrap=True)
                axes[i].axis('off')
                
            except Exception as e:
                axes[i].text(0.5, 0.5, f"Error rendering {config_name}:\n{str(e)}", 
                           ha='center', va='center', transform=axes[i].transAxes)
                axes[i].set_title(f"{config_name} (Error)")
                axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()

# Plot images for the first example
print("🖼️ Rendered Images Comparison (Example 1)")
plot_rendered_images(datasets, example_idx=0, max_configs=4)

In [ ]:
# Plot images for another example
print("🖼️ Rendered Images Comparison (Example 2)")
plot_rendered_images(datasets, example_idx=1, max_configs=4)